<a href="https://colab.research.google.com/github/25CristopherJoshuaReyesGtz1405/MachineLearning/blob/main/Machine_Learning_Y_Deep_Learning_Pr%C3%A1ctica_No_5_Unidad_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![TecNM](https://drive.google.com/uc?id=1HP9R3V9XnozjtY08gGdKhoCKo76MKgLr)

---
<div align="center">

  ## **Machine Learning y Deep Learning**

  ### **Unidad 2 - Modelos de Aprendizaje Supervisado**

  #### **Práctica No. 5 - Modelado Robusto mediante Máquinas de Soporte para Regresión (SVR)**

  ##### **Docente:** Dr. José Gabriel Rodríguez Rivas
  ##### **Estudiante:** Cristopher Joshua Reyes Gutiérrez

</div>

---

### **1. Introducción**

En el estudio avanzado de la Inteligencia Artificial, nos enfrentamos constantemente al reto de construir modelos que no solo posean una alta capacidad predictiva, sino que también demuestren resiliencia ante la presencia de ruido y valores atípicos. Mientras que los modelos de ensamble como el Random Forest buscan la precisión mediante el promedio de múltiples decisiones, las **Máquinas de Soporte para Regresión (SVR)** proponen un enfoque basado en la optimización del margen y la minimización del riesgo estructural.

La SVR es una adaptación de las potentes Máquinas de Soporte Vectorial (SVM) diseñada específicamente para variables continuas. Su filosofía disruptiva no busca minimizar el error global de forma tradicional; en su lugar, intenta encontrar una función que se ajuste a los datos dentro de una franja de tolerancia específica. Esta práctica representa la culminación de nuestra investigación sobre modelos de regresión, explorando cómo la proyección de datos hacia espacios de mayor dimensión nos permite capturar tendencias no lineales con una estabilidad estadística superior a la de los métodos convencionales.

### **2. Objetivos**

#### **2.1 Objetivo General**
Analizar, implementar y validar el rendimiento de las Máquinas de Soporte para Regresión (SVR) en la estimación de precios de automóviles, examinando el impacto crítico del escalamiento de datos y la optimización de hiperparámetros en la construcción de modelos robustos.

#### **2.2 Objetivos Específicos**
* **Establecer** un protocolo de preprocesamiento avanzado que incluya el escalamiento estándar tanto de las variables predictoras como de la variable objetivo, dada la naturaleza geométrica del algoritmo.
* **Evaluar** la efectividad del kernel de Función de Base Radial (RBF) para transformar el espacio de características y capturar relaciones no lineales complejas en el mercado automotriz.
* **Optimizar** el modelo mediante el ajuste fino de la constante de penalización ($C$), el margen de tolerancia ($\epsilon$) y el parámetro de influencia ($\gamma$), buscando el equilibrio ideal entre suavidad y precisión.
* **Contrastar** visualmente la capacidad de generalización del modelo frente a los datos reales, identificando la precisión del "tubo de insensibilidad" en la captura de la tendencia central de los precios.

### **3. Antecedentes Históricos -  La Geometría de la SVR**

La SVR se fundamenta en la búsqueda de un hiperplano en un espacio de características de alta dimensión, definido por un conjunto de puntos críticos conocidos como vectores de soporte. A diferencia de la regresión de mínimos cuadrados, la SVR se enfoca en la planitud del modelo.

#### **3.1 El Tubo de $\epsilon$-Insensibilidad**
El concepto central de este modelo es el margen de tolerancia $\epsilon$. El algoritmo busca una función que se desvíe como máximo $\epsilon$ de los valores reales para cada punto de entrenamiento. Los errores que ocurren dentro de este "tubo" no son penalizados, lo que permite al modelo ignorar fluctuaciones menores y concentrarse en la estructura robusta de los datos. Solo los puntos que caen fuera de este margen —los vectores de soporte— son los que realmente dictan la posición y forma del hiperplano predictivo.



#### **3.2 El Poder de los Kernels y la Dimensión Superior**
Para abordar la no linealidad, la SVR utiliza el "truco del kernel". Mediante funciones como el **Radial Basis Function (RBF)**, el modelo proyecta los datos originales de entrada hacia un espacio de dimensiones superiores donde una separación lineal es posible. Esto permite que el modelo resuelva problemas de alta complejidad sin necesidad de calcular explícitamente las coordenadas en dicho espacio, manteniendo una eficiencia computacional notable frente a otros métodos no paramétricos.

## **4. Desarrollo Metodológico**

La implementación de la SVR requiere un flujo de trabajo meticuloso, donde el tratamiento de las magnitudes de los datos es el paso más determinante para el éxito del entrenamiento.

### **4.1 Fase I: Configuración del Ecosistema**
Dado que la SVR basa su aprendizaje en el cálculo de distancias euclidianas, las variables con escalas mayores podrían dominar el modelo injustamente. Por ello, aplicamos un escalamiento estándar integral utilizando la clase `StandardScaler` tanto para el vector de características ($X$) como para la variable objetivo ($y$).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Carga del conjunto de datos preprocesado
df = pd.read_csv("autos2.csv")


####  **4.1.1 Fase I.I: Escalamiento de Datos**

In [ ]:
# Selección de dimensiones técnicas
X = df[['horsepower', 'engine-size', 'city-mpg', 'wheel-base', 'bore']]
y = df['price']

# Partición 80/20 para entrenamiento y validación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# FASE DE ESCALAMIENTO (Determinante para SVR)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Escalamiento del target y transformación a vector plano
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

### **4.2 Fase II: Definición y Entrenamiento del Modelo SVR**
Utilizamos el kernel RBF para gestionar la complejidad no lineal detectada en prácticas anteriores. El entrenamiento se realiza bajo una configuración de parámetros que equilibra la tolerancia al error y la suavidad del hiperplano.

### **4.3 Fase III: Inferencia y Desescalamiento**
Una vez generadas las predicciones en la escala estándar, es necesario revertir la transformación para interpretar los resultados en las unidades originales de moneda (dólares).

In [ ]:
# Inicialización y ajuste del modelo SVR
# Parámetros optimizados tras validación cruzada: C=10, gamma=0.1, epsilon=0.01
svr_model = SVR(kernel='rbf', C=10, gamma=0.1, epsilon=0.01)
svr_model.fit(X_train_scaled, y_train_scaled)

# Generación de predicciones
y_pred_scaled = svr_model.predict(X_test_scaled)

# Inversión de la transformación para análisis en unidades reales
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

### **4.4 Fase IV: Evaluación Multimétrica**
Cuantificamos el desempeño mediante el Error Cuadrático Medio y el Coeficiente de Determinación, contrastando la precisión alcanzada por este modelo robusto frente a los algoritmos de ensamble previos.

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Resultados de Evaluación SVR:")
print(f"R2 Score (Bondad de Ajuste): {r2:.2f}")
print(f"RMSE (Desviación en $): {rmse:.2f}")

## **5. Diagnóstico de Resultados**

### **5.1 Análisis de Densidad de Predicción**
Visualizamos la capacidad del modelo para ajustarse a la distribución bimodal del mercado automotriz. El gráfico de densidad permite observar si el SVR logra mantenerse dentro del rango real de precios sin ser arrastrado por valores extremos.

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(y_test, label='Precios Reales', linewidth=2, color='darkblue')
sns.kdeplot(y_pred, label='Precios Predichos (SVR)', linewidth=2, linestyle='--', color='darkgreen')
plt.title("SVR: Comparativa de Distribución Real vs. Predicha", fontsize=14)
plt.xlabel("Precio ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## **6. Conclusiones y Análisis Crítico**

Al finalizar esta quinta etapa experimental con Máquinas de Soporte para Regresión, se establecen las siguientes conclusiones:

1.  **Robustez Estadística:** El modelo SVR alcanzó un $R^2$ de **0.81**, logrando capturar el **81%** de la varianza del precio. Si bien es ligeramente menor a la precisión del Random Forest, el SVR demuestra una mayor estabilidad estructural al ignorar errores pequeños dentro de su margen de tolerancia.
2.  **Eficacia del Escalamiento:** Se confirmó empíricamente que la SVR no puede operar de forma efectiva sin una estandarización previa. El éxito del modelo radica en que todas las variables técnicas fueron tratadas bajo una escala uniforme, permitiendo que el cálculo de los vectores de soporte fuera equitativo.
3.  **Gestión de la No Linealidad:** El uso del kernel RBF fue fundamental para superar las barreras de los modelos lineales iniciales, permitiendo proyecciones que capturan la tendencia central de los datos con un RMSE de aproximadamente **4,864.03**.
4.  **Balance de Parámetros:** Se observó que una constante $C$ moderada permite un modelo con buena capacidad de generalización, evitando que el SVR se vuelva demasiado sensible al ruido del dataset de entrenamiento.

---

## **7. Referencias Bibliográficas**

* **Breiman, L. (2001).** *Random forests. Machine Learning*, 45(1), 5-32.
* **Montgomery, D. C., et al. (2003).** *Introducción al análisis de regresión lineal*. CECSA.
* **Smola, A. J., & Schölkopf, B. (2004).** *A tutorial on support vector regression*. Statistics and Computing.
* **Vapnik, V. N. (2000).** *The nature of statistical learning theory*. Springer.